# Counterfactual

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google/yggdrasil-decision-forests/blob/main/documentation/public/docs/tutorial/counterfactual.ipynb)

## Setup


In [ ]:
pip install ydf scikit-learn umap-learn plotly -U -q

In [2]:
import numpy as np
import pandas as pd  # We use Pandas to load small datasets
import ydf  # Yggdrasil Decision Forests

## What are neighbor examples and counterfactual examples?

**Neighbor examples** are examples that are similar to one another according to the model, that is, examples where the model gives the same predictions for the same reasons.
Looking at the similarities and differences between the neighbors of an example is a great way to understand the predictions of the model for this example

**Counterfactual examples** are neighbor examples that don't have the same labels as an example of interest. When the model predictions are unexpected, looking at its counterfactual examples is a great way to understand why.

## What is an example distance?

The distance between two examples is a numerical value between 0 and 1 indicating how different two examples are. The neighbors of an example of interest are the examples with the smallest distances.

Decision forest models define an **implicit measure of proximity or similarity between two examples**, referred to as **distance**. The distance represents how two examples are treated similarly in the model. Informally, **two examples are close if they are of the same class and for the same reasons**.

This distance is useful for understanding models and their predictions. For example, we can use it for clustering, manifold learning, or simply to look at the training examples that are nearest to a test example (called **counterfactual examples**). This can help us to understand why the model made its predictions.

Keep in mind that a decision forest's distance measure is just one of many reasonable distance metrics on a dataset. One of its many advantages is that allows comparing features on different scales and with different semantics.

In this notebook, we will train a model and use its distance to:

- Find training examples that are neighbors of a test example and use them to explain the model's predictions.

- Map all the examples onto an interactive two-dimensional plot (also known as a 2D manifold) and automatically detect two-dimensional clusters of examples that behave similarly.

- Apply hierarchical clustering to explain how the model works as a whole.

**The More You Know:** [Leo Breiman](https://en.wikipedia.org/wiki/Leo_Breiman), the author of the [random forest](https://developers.google.com/machine-learning/glossary#random-forest) learning algorithm, [proposed](https://www.stat.berkeley.edu/~breiman/RandomForests/cc_home.htm#prox) a method to measure the *proximity* between two examples using a pre-trained Random Forest (RF) model. He qualifies this method as <i>"[...] one of the most useful tools in random forests."</i>. When using Random Forest models, this is the distance used by YDF.


## Find closest training examples to a test example

Let's download a classification dataset.

In [3]:
ds_path = "https://raw.githubusercontent.com/google/yggdrasil-decision-forests/main/yggdrasil_decision_forests/test_data/dataset"
train_ds = pd.read_csv(f"{ds_path}/adult_train.csv")
test_ds = pd.read_csv(f"{ds_path}/adult_test.csv")

# Print the first 5 training examples
train_ds.head(5)

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,44,Private,228057,7th-8th,4,Married-civ-spouse,Machine-op-inspct,Wife,White,Female,0,0,40,Dominican-Republic,<=50K
1,20,Private,299047,Some-college,10,Never-married,Other-service,Not-in-family,White,Female,0,0,20,United-States,<=50K
2,40,Private,342164,HS-grad,9,Separated,Adm-clerical,Unmarried,White,Female,0,0,37,United-States,<=50K
3,30,Private,361742,Some-college,10,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,50,United-States,<=50K
4,67,Self-emp-inc,171564,HS-grad,9,Married-civ-spouse,Prof-specialty,Wife,White,Female,20051,0,30,England,>50K


We train a random forest on this dataset.

In [4]:
model = ydf.RandomForestLearner(label="income").train(train_ds)

Train model on 22792 examples
Model trained in 0:00:01.086534


We need to select a example to explain. Let's select the first example of the testing dataset.

In [5]:
selected_example_idx = 0  # Change to select another example
selected_example = test_ds[selected_example_idx : (selected_example_idx + 1)]
selected_example

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K


On this example, the model predicts:

In [6]:
model.predict(selected_example)

array([0.01333333], dtype=float32)

In other words, the negative class `<=50K` with $1-0.01=99\%$ probability.

Now, we compute the distance between the selected test example and all the training examples.

In [7]:
distances = model.distance(train_ds, selected_example).squeeze()

print("distances:", distances)

distances: [1.         1.         0.99666667 ... 0.99333334 0.99666667 1.        ]


Let's find the the five training examples with smallest distance to our chosen example.

In [8]:
close_train_idxs = np.argsort(distances)[:5]
print("close_train_idxs:", close_train_idxs)

print("Selected test examples:")
train_ds.iloc[close_train_idxs]

close_train_idxs: [16596 21845 10321  7299 20945]
Selected test examples:


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
16596,41,State-gov,26892,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,0,0,40,United-States,<=50K
21845,37,State-gov,60227,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,0,0,38,United-States,<=50K
10321,40,Private,82161,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,0,0,40,United-States,<=50K
7299,30,State-gov,158291,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,0,0,40,United-States,<=50K
20945,44,State-gov,296326,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,0,0,40,United-States,<=50K


**Observations:**

- For the chosen example, the model predicted class `<=50K`. For the five closes examples, the model had the same prediction.
- The closest examples share many features values, such as `education`, `marital status`, `occupation`, `race`, and working between 37 and 40 `hours per week`. This explains well why these examples are close to each other.
- The examples' `age`s range between 30 and 40, meaning the model sees this age range as equivalent for those examples.


## Two dimensional projections of the examples

Our first use of the proximity is to project the examples on the two dimensional plane.

For that, we can use [t-SNE](https://en.wikipedia.org/wiki/T-distributed_stochastic_neighbor_embedding) or [UMAP](https://umap-learn.readthedocs.io).

In [9]:
import IPython
import plotly.graph_objs as go
import plotly.offline

In [10]:
# Pairwise distance between all testing examples
distances = model.distance(test_ds, test_ds)

In [11]:
# Organize the examples on a 2D manifold.

# Select the method you want to use.
# Using different methods and different parameters will change the projection.
manifold_lib = "UMAP"  # "UMAP" or "TSNE"

if manifold_lib == "TSNE":
  from sklearn.manifold import TSNE

  manifold = TSNE(
      # Number of dimensions to display. 3d is also possible.
      n_components=2,
      # Control the shape of the projection. Higher values create more
      # distinct but also more collapsed clusters. Can be in 5-50.
      perplexity=20,
      metric="precomputed",
      init="random",
      verbose=1,
      learning_rate="auto",
  ).fit_transform(distances)
elif manifold_lib == "UMAP":
  import umap

  manifold = umap.UMAP(
      # Number of dimensions to display. 3d is also possible.
      n_components=2,
      # Balance local versus global structure in the data.
      n_neighbors=15,
      metric="precomputed",
  ).fit_transform(distances)
else:
  raise ValueError(f"Unknown lib: {manifold_lib}")

Let's create an interactive plot with the example features.

In [12]:
def example_to_html(example):
  return "<br>".join([f"<b>{k}:</b> {v}" for k, v in example.items()])


def interactive_plot(dataset, projections, num_points=2000):
  # Plot a random sample of the examples to keep the plot light.
  rng = np.random.default_rng(seed=1)
  idxs = rng.choice(
      len(dataset), size=min(num_points, len(dataset)), replace=False
  )
  dataset = dataset.iloc[idxs]
  projections = projections[idxs]

  colors = (dataset["income"] == ">50K").map(lambda x: ["red", "blue"][x])
  labels = list(dataset.apply(example_to_html, axis=1).values)
  fig = go.Figure(
      data=[
          go.Scatter(
              x=projections[:, 0],
              y=projections[:, 1],
              text=labels,
              mode="markers",
              marker={"color": colors, "size": 3},
          )
      ],
      layout=go.Layout(width=500, height=500, template="simple_white"),
  )
  # Self-contained HTML, rendered both in notebooks and in the documentation.
  # Note: plotly.js is loaded from an explicit URL rather than with
  # `include_plotlyjs="cdn"`, whose integrity hash may not match the CDN file.
  plotly_js_url = (
      "https://cdn.jsdelivr.net/npm/plotly.js-dist-min@"
      f"{plotly.offline.get_plotlyjs_version()}/plotly.min.js"
  )
  IPython.display.display(
      IPython.display.HTML(
          fig.to_html(include_plotlyjs=plotly_js_url, full_html=False)
      )
  )


interactive_plot(test_ds, manifold)

**Note:** Move your mouse over the plot to see the values of the examples.

The colors represent the labels. We can see clusters of uniform colors (clusters where all the labels are the same), and clusters of mixed colors (clusters where the model has difficulty making good predictions).

Can you make sense of those clusters?


## Cluster examples

We can also cluster examples. [Many methods](https://scikit-learn.org/stable/modules/clustering.html) are available. Let's use `AgglomerativeClustering`.

In [13]:
from sklearn.cluster import AgglomerativeClustering

num_clusters = 6
clustering = AgglomerativeClustering(
    n_clusters=num_clusters,
    metric="precomputed",
    linkage="average",
).fit(distances)

Next, we print the statistics of the features and one example in each cluster.

In [14]:
import IPython

for cluster_idx in range(num_clusters):
  selected_examples = test_ds[clustering.labels_ == cluster_idx]
  print(f"Cluster #{cluster_idx} with {len(selected_examples)} examples")
  print("=============================")
  IPython.display.display(selected_examples.describe())
  IPython.display.display(selected_examples.iloc[:1])

Cluster #0 with 5169 examples


,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,5169.000000,5.169000e+03,5169.000000,5169.000000,5169.000000,5169.000000
mean,34.156317,1.930069e+05,9.778100,104.245309,58.859934,37.877539
std,13.421280,1.053680e+05,2.449272,641.538542,333.039858,12.445170
min,17.000000,1.921400e+04,1.000000,0.000000,0.000000,1.000000
25%,23.000000,1.205440e+05,9.000000,0.000000,0.000000,35.000000
50%,31.000000,1.815530e+05,10.000000,0.000000,0.000000,40.000000
75%,42.000000,2.413500e+05,11.000000,0.000000,0.000000,40.000000
max,90.000000,1.038553e+06,16.000000,6849.000000,3770.000000,99.000000


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K


Cluster #1 with 2436 examples


,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,2436.000000,2436.000000,2436.000000,2436.000000,2436.000000,2436.000000
mean,42.034072,183025.072250,9.591544,208.834565,30.669540,42.964286
std,11.736571,98692.366538,0.833731,879.124846,221.591421,11.924056
min,18.000000,19395.000000,6.000000,0.000000,0.000000,2.000000
25%,33.000000,115015.750000,9.000000,0.000000,0.000000,40.000000
50%,41.000000,175634.000000,9.000000,0.000000,0.000000,40.000000
75%,50.000000,229839.750000,10.000000,0.000000,0.000000,48.000000
max,90.000000,671292.000000,12.000000,5013.000000,1740.000000,99.000000


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
1,40,Private,121772,Assoc-voc,11,Married-civ-spouse,Craft-repair,Husband,Asian-Pac-Islander,Male,0,0,40,NaN,>50K


... (8 outputs left out)

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
25,48,Self-emp-not-inc,191277,Doctorate,16,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,1902,60,United-States,>50K


Cluster #5 with 482 examples


,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,482.000000,482.000000,482.000000,482.000000,482.0,482.000000
mean,45.599585,191533.381743,11.802905,19079.352697,0.0,46.622407
std,11.009325,103712.256794,2.506655,25850.885498,0.0,11.639712
min,20.000000,19302.000000,1.000000,5178.000000,0.0,2.000000
25%,38.000000,119820.750000,10.000000,7331.000000,0.0,40.000000
50%,44.000000,175442.500000,13.000000,9953.000000,0.0,45.000000
75%,52.000000,237169.000000,14.000000,15024.000000,0.0,50.000000
max,78.000000,617021.000000,16.000000,99999.000000,0.0,99.000000


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
21,44,Private,343591,HS-grad,9,Divorced,Craft-repair,Not-in-family,White,Female,14344,0,40,United-States,>50K
